In [1]:
import sys 
import os
import yaml
current_dir = os.path.dirname(os.path.abspath('.'))
project_root = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.insert(0, project_root)

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
plt.style.use('ggplot')

In [3]:
from functions.make_dataset import split_data, save_data
from Titanic.src.features.feature_eng import PreprocessingOrchestrator

In [4]:
# 1. Carregar configurações
with open(os.path.join(project_root, "Titanic/config/config.yaml"), "r") as f:
    config = yaml.safe_load(f)
    
# pipeline selection    
with open(os.path.join(project_root, "Titanic/config/pipeline.yaml"), "r") as f:
    config_pipe = yaml.safe_load(f)

# Read Dataset

In [5]:
df = pd.read_parquet(
        os.path.join(
            config['init_path'],
            config['data']['processed'],
            "train_features.parquet"
            )       
        )

In [6]:
print(f'Dataset rows and columns: {df.shape}')

Dataset rows and columns: (891, 17)


In [7]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Ticket_1p,Cabin_1p,FamilySize,IsAlone
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1.0,0.0,A521171,7.2500,None,S,Mr,A,None,2,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1.0,0.0,PC17599,71.2833,C85,C,Mrs,P,C,2,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0.0,0.0,STONO23101282,7.9250,None,S,Miss,S,None,1,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1.0,0.0,113803,53.1000,C123,S,Mrs,1,C,2,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0.0,0.0,373450,8.0500,None,S,Mr,3,None,1,1


In [8]:
 # 2. Processamento de dados    
X_train, X_val, y_train, y_val = split_data(
        df, 
        target_column=config_pipe['features']['target'][0]
        )

In [9]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Ticket_1p,Cabin_1p,FamilySize,IsAlone
331,332,1,"Partner, Mr. Austen",male,45.5,0.0,0.0,113043,28.5000,C124,S,Mr,1,C,1,1
733,734,2,"Berriman, Mr. William John",male,23.0,0.0,0.0,28425,13.0000,None,S,Mr,2,None,1,1
382,383,3,"Tikkanen, Mr. Juho",male,32.0,0.0,0.0,STONO23101293,7.9250,None,S,Mr,S,None,1,1
704,705,3,"Hansen, Mr. Henrik Juul",male,26.0,1.0,0.0,350025,7.8542,None,S,Mr,3,None,2,0
813,814,3,"Andersson, Miss. Ebba Iris Alfrida",female,6.0,4.0,2.0,347082,31.2750,None,S,Miss,3,None,7,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,107,3,"Salkjelsvik, Miss. Anna Kristine",female,21.0,0.0,0.0,343120,7.6500,None,S,Miss,3,None,1,1
270,271,1,"Cairns, Mr. Alexander",male,NaN,0.0,0.0,113798,31.0000,None,S,Mr,1,None,1,1
860,861,3,"Hansen, Mr. Claus Peter",male,41.0,2.0,0.0,350026,14.1083,None,S,Mr,3,None,3,0
435,436,1,"Carter, Miss. Lucile Polk",female,14.0,1.0,2.0,113760,120.0000,B96B98,S,Miss,1,B,4,0


In [10]:
# 3. Feature Engineering        
preprocessor = PreprocessingOrchestrator(
        numerical_con=config_pipe['features']['num_con'], 
        numerical_dis=config_pipe['features']['num_dis'], 
        categorical_var=config_pipe['features']['cat_var'])


In [11]:
config_pipe['features']['num_con']

['Age', 'Fare']

In [12]:
config_pipe['features']['num_dis']

['IsAlone', 'FamilySize']

In [13]:
config_pipe['features']['cat_var']

['Pclass', 'Sex', 'Embarked', 'Title', 'Cabin_1p']

In [14]:
    # define pipiline
pipeline_name = "Pipeline3"
    
pipe = preprocessor.apply(pipeline_name)    
X_train = pipe.fit_transform(X_train, y_train)
X_val = pipe.transform(X_val)    


c:\Users\gustavo\anaconda3\envs\mf_tf\lib\site-packages\feature_engine\encoding\rare_label.py:216: UserWarning: The number of unique categories for variable Sex is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(


In [20]:
X_train

,numerical_pipe_con__Age,numerical_pipe_con__Fare,numerical_pipe_dis__SibSp,numerical_pipe_dis__Parch,numerical_pipe_dis__IsAlone,numerical_pipe_dis__FamilySize,categorical_pipe__Pclass,categorical_pipe__Sex,categorical_pipe__Embarked,categorical_pipe__Title,categorical_pipe__Cabin_1p
331,2,1,0,0.0,1,0,2,0,0,0,4
733,2,0,0,0.0,1,0,1,0,0,0,1
382,2,0,0,0.0,1,0,0,0,0,0,1
704,2,0,0,0.0,0,0,0,0,0,0,1
813,0,1,0,2.0,0,1,0,1,0,3,1
...,...,...,...,...,...,...,...,...,...,...,...
106,2,0,0,0.0,1,0,0,1,0,3,1
270,2,1,0,0.0,1,0,2,0,0,0,1
860,2,0,0,0.0,0,0,0,0,0,0,1
435,1,3,0,2.0,0,0,2,1,0,3,5


In [17]:
from sklearn.model_selection import GridSearchCV
# Define the parameter grid to search over
param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Create a RandomForestClassifier instance
rf = RandomForestClassifier(random_state=42)

# Setup the RandomizedSearchCV instance
random_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, verbose=0, n_jobs=-1)

# Fit the RandomizedSearchCV instance to the data
random_search.fit(X_train, y_train)

# Print the best parameters and the corresponding score
print(f"Best parameters: {random_search.best_params_}")
print(f"Best score: {random_search.best_score_}")

# Retrieve the best estimator
best_rf = random_search.best_estimator_

Best parameters: {'bootstrap': True, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
Best score: 0.8314777387748348


In [18]:
best_rf.score(X_train, y_train)

0.8848314606741573

In [19]:
# Evaluate the model on the test set and generate all classification metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Make predictions on the test set
y_pred = best_rf.predict(X_val)

# Generate the classification report
report = classification_report(y_val, y_pred)  # Renamed variable to avoid conflict

# Generate the confusion matrix
matrix = confusion_matrix(y_val, y_pred)  # Renamed variable to avoid conflict

# Generate the accuracy score
accuracy = accuracy_score(y_val, y_pred)  # Renamed variable to avoid conflict

print(f"Classification Report:\n{report}\n")
print(f"Confusion Matrix:\n{matrix}\n")
print(f"Accuracy Score: {accuracy}\n")

# Make predictions on the test set (assuming test_ds is prepared correctly)
# test_predictions = best_rf.predict(test_ds)
# Now you can use test_predictions for further analysis or export

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.85      0.84       105
           1       0.78      0.76      0.77        74

    accuracy                           0.81       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179


Confusion Matrix:
[[89 16]
 [18 56]]

Accuracy Score: 0.8100558659217877

